# Firestore Database Setup

This notebook demonstrates how to:
1. Enable the Firestore API
2. Create a Firestore database in Native Mode
3. Verify the setup with basic operations

**Firestore Mode**: Native Mode - Document database with real-time updates and offline support

**Location**: Single region (us-central1) for this demo

**Estimated Time**: 5-8 minutes (including database provisioning which takes 2-3 minutes)

**Required Permissions**:
- `roles/datastore.owner` (to create and manage Firestore databases)

**Cost Considerations**:

Firestore offers a generous free tier:
- 1 GB storage
- 50,000 document reads per day
- 20,000 document writes per day
- 20,000 document deletes per day

This basic setup will remain well within free tier limits.

## Section 1: Configuration & Authentication

In [1]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
DATABASE_ID = "(default)"  # Use default Firestore database

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Database ID: {DATABASE_ID}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Database ID: (default)


In [2]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: johnswain-1200-20260303091357


In [3]:
# Check required IAM permissions
print("🔐 Checking required IAM permissions...\n")
print("This notebook requires the following permissions:\n")
print("   1️⃣  Firestore:")
print("      - roles/datastore.owner (to create and manage Firestore)\n")

# Try to get user email
try:
    if hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    elif hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    else:
        user_email = "(Unable to detect email from credentials)"
except:
    user_email = "(Unable to detect email from credentials)"

print(f"   Your account: {user_email}\n")
print("💡 To grant these permissions, run these commands in your terminal:\n")
print("   # Grant Datastore Owner")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/datastore.owner'\n")

🔐 Checking required IAM permissions...

This notebook requires the following permissions:

   1️⃣  Firestore:
      - roles/datastore.owner (to create and manage Firestore)

   Your account: (Unable to detect email from credentials)

💡 To grant these permissions, run these commands in your terminal:

   # Grant Datastore Owner
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/datastore.owner'



In [4]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = [
    "google-cloud-firestore"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

📦 Installing required libraries...
✅ Libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# Import libraries
import time
import requests
from google.auth.transport.requests import Request
from google.cloud import firestore

# Refresh credentials for REST API calls
credentials.refresh(Request())

print("✅ Libraries imported and credentials refreshed")

✅ Libraries imported and credentials refreshed


## Section 2: Enable Required APIs

Enable the Firestore API for the project.

In [6]:
# Check if Firestore API is enabled
print("🔍 Checking Firestore API status...")

api_url = f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/firestore.googleapis.com"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

response = requests.get(api_url, headers=headers)

if response.status_code == 200:
    service_info = response.json()
    state = service_info.get('state', 'UNKNOWN')
    
    if state == 'ENABLED':
        print("✅ Firestore API is already enabled")
        API_ENABLED = True
    else:
        print(f"ℹ️  Firestore API state: {state}")
        API_ENABLED = False
else:
    print(f"⚠️  Could not check API status: {response.status_code}")
    API_ENABLED = False

🔍 Checking Firestore API status...
ℹ️  Firestore API state: DISABLED


In [7]:
# Enable Firestore API if needed
if not API_ENABLED:
    print("🚀 Enabling Firestore API...")
    
    enable_url = f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/firestore.googleapis.com:enable"
    
    response = requests.post(enable_url, headers=headers)
    
    if response.status_code == 200:
        print("✅ Firestore API enabled successfully")
        print("⏳ Waiting for API to become fully available...")
        time.sleep(10)  # Wait for API to propagate
        API_ENABLED = True
    else:
        print(f"❌ Failed to enable Firestore API: {response.status_code}")
        print(response.text)
        print("\n💡 Try enabling it manually:")
        print(f"   gcloud services enable firestore.googleapis.com --project={PROJECT_ID}")
else:
    print("ℹ️  Skipping API enablement - already enabled")

🚀 Enabling Firestore API...
✅ Firestore API enabled successfully
⏳ Waiting for API to become fully available...


## Section 3: Create Firestore Database

This section creates a Firestore database in Native Mode using the Firestore Admin API.

**Database Configuration:**
- Type: Native Mode (for real-time and mobile applications)
- Location: us-central1 (single region)
- Database ID: (default)

**Note**: Database creation takes approximately 2-3 minutes.

In [8]:
# Check if Firestore database already exists
print("🔍 Checking if Firestore database exists...")

# For the default database, we need to check using Firestore Admin API
db_url = f"https://firestore.googleapis.com/v1/projects/{PROJECT_ID}/databases/{DATABASE_ID}"

response = requests.get(db_url, headers=headers)

if response.status_code == 200:
    db_info = response.json()
    print(f"✅ Firestore database '{DATABASE_ID}' already exists")
    print(f"   Name: {db_info.get('name', 'N/A')}")
    print(f"   Type: {db_info.get('type', 'N/A')}")
    print(f"   Location: {db_info.get('locationId', 'N/A')}")
    DATABASE_EXISTS = True
elif response.status_code == 404:
    print(f"ℹ️  Firestore database '{DATABASE_ID}' does not exist yet")
    DATABASE_EXISTS = False
else:
    print(f"⚠️  Error checking database: {response.status_code}")
    print(response.text)
    DATABASE_EXISTS = False

🔍 Checking if Firestore database exists...
ℹ️  Firestore database '(default)' does not exist yet


In [9]:
# Create Firestore database if it doesn't exist
if not DATABASE_EXISTS:
    print(f"🚀 Creating Firestore database '{DATABASE_ID}'...")
    print("⏱️  This will take approximately 2-3 minutes")
    
    create_url = f"https://firestore.googleapis.com/v1/projects/{PROJECT_ID}/databases"
    
    database_config = {
        "name": f"projects/{PROJECT_ID}/databases/{DATABASE_ID}",
        "locationId": REGION,
        "type": "FIRESTORE_NATIVE"
    }
    
    # Note: databaseId parameter goes as query string
    create_url_with_id = f"{create_url}?databaseId={DATABASE_ID}"
    
    response = requests.post(create_url_with_id, headers=headers, json=database_config)
    
    if response.status_code in [200, 201]:
        print("✅ Database creation started")
        operation = response.json()
        operation_name = operation.get('name')
        
        if operation_name:
            print(f"   Operation: {operation_name}")
            
            # Wait for operation to complete
            print("\n⏳ Waiting for database creation to complete...")
            max_wait_time = 300  # 5 minutes
            start_time = time.time()
            
            while time.time() - start_time < max_wait_time:
                op_url = f"https://firestore.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    
                    if op_data.get('done', False):
                        if 'error' in op_data:
                            print(f"\n❌ Database creation failed: {op_data['error']}")
                            break
                        else:
                            elapsed = int(time.time() - start_time)
                            print(f"\n✅ Database created successfully! (took {elapsed}s)")
                            DATABASE_EXISTS = True
                            break
                    else:
                        elapsed = int(time.time() - start_time)
                        print(f"   Waiting... {elapsed}s elapsed", end='\r')
                
                time.sleep(5)  # Check every 5 seconds
            
            if not DATABASE_EXISTS:
                elapsed = int(time.time() - start_time)
                print(f"\n⚠️  Database creation timed out after {elapsed}s")
                print("   Please check the GCP Console to verify database status.")
        else:
            # Response might be the database itself (immediate creation)
            print("✅ Database created successfully!")
            DATABASE_EXISTS = True
    else:
        print(f"❌ Failed to create database: {response.status_code}")
        print(response.text)
        print("\n💡 You may need to create the database using the console:")
        print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
else:
    print("ℹ️  Skipping database creation - database already exists")

🚀 Creating Firestore database '(default)'...
⏱️  This will take approximately 2-3 minutes
✅ Database creation started
   Operation: projects/johnswain-1200-20260303091357/databases/(default)/operations/1WC0sGxIIaexQGr8k0IlBxAqMWxhcnRuZWMtc3ULIgUQAc-xvKgQBs28s-kIDAovGg

⏳ Waiting for database creation to complete...

✅ Database created successfully! (took 1s)


In [10]:
# Verify database status
if DATABASE_EXISTS:
    print("\n🔍 Verifying database status...")
    
    response = requests.get(db_url, headers=headers)
    
    if response.status_code == 200:
        db_info = response.json()
        print("\n📊 Database Information:")
        print(f"   Name: {db_info.get('name', 'N/A')}")
        print(f"   Type: {db_info.get('type', 'N/A')}")
        print(f"   Location: {db_info.get('locationId', 'N/A')}")
        print(f"\n🔗 View in Console:")
        print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
    else:
        print(f"⚠️  Could not verify database: {response.status_code}")
        print(response.text)


🔍 Verifying database status...

📊 Database Information:
   Name: projects/johnswain-1200-20260303091357/databases/(default)
   Type: FIRESTORE_NATIVE
   Location: us-central1

🔗 View in Console:
   https://console.cloud.google.com/firestore/databases?project=johnswain-1200-20260303091357


## Section 4: Verify Firestore Setup

Test the Firestore setup with basic create, read, and delete operations.

In [11]:
# Initialize Firestore client
print("🔌 Initializing Firestore client...")

try:
    db = firestore.Client(project=PROJECT_ID, database=DATABASE_ID)
    print("✅ Firestore client initialized successfully")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Database: {DATABASE_ID}")
except Exception as e:
    print(f"❌ Failed to initialize Firestore client: {e}")
    print("\n💡 Make sure the Firestore API is enabled and you have proper permissions.")
    raise

🔌 Initializing Firestore client...
✅ Firestore client initialized successfully
   Project: johnswain-1200-20260303091357
   Database: (default)


In [12]:
# Test: Create a document
print("\n📝 Testing: Create a test document...")

test_collection = "test_setup"
test_doc_id = "verification_doc"

test_data = {
    "message": "Hello from Firestore!",
    "timestamp": firestore.SERVER_TIMESTAMP,
    "test_number": 42,
    "test_boolean": True,
    "nested_data": {
        "field1": "value1",
        "field2": "value2"
    }
}

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc_ref.set(test_data)
    print(f"✅ Document created: {test_collection}/{test_doc_id}")
except Exception as e:
    print(f"❌ Failed to create document: {e}")
    raise


📝 Testing: Create a test document...
✅ Document created: test_setup/verification_doc


In [13]:
# Test: Read the document
print("\n📖 Testing: Read the test document...")

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc = doc_ref.get()
    
    if doc.exists:
        print(f"✅ Document retrieved successfully")
        print(f"\n   Document data:")
        for key, value in doc.to_dict().items():
            print(f"     {key}: {value}")
    else:
        print(f"❌ Document does not exist")
except Exception as e:
    print(f"❌ Failed to read document: {e}")
    raise


📖 Testing: Read the test document...
✅ Document retrieved successfully

   Document data:
     timestamp: 2026-03-09 19:05:38.806000+00:00
     test_number: 42
     message: Hello from Firestore!
     nested_data: {'field1': 'value1', 'field2': 'value2'}
     test_boolean: True


In [14]:
# Test: Query the collection
print("\n🔍 Testing: Query the collection...")

try:
    docs = db.collection(test_collection).limit(5).stream()
    
    doc_count = 0
    for doc in docs:
        doc_count += 1
        print(f"   Document {doc.id}: {doc.to_dict().get('message', 'N/A')}")
    
    print(f"\n✅ Query successful - found {doc_count} document(s)")
except Exception as e:
    print(f"❌ Failed to query collection: {e}")
    raise


🔍 Testing: Query the collection...
   Document verification_doc: Hello from Firestore!

✅ Query successful - found 1 document(s)


In [15]:
# Test: Delete the test document
print("\n🗑️  Testing: Delete the test document...")

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc_ref.delete()
    print(f"✅ Document deleted: {test_collection}/{test_doc_id}")
    
    # Verify deletion
    doc = doc_ref.get()
    if not doc.exists:
        print("✅ Deletion verified - document no longer exists")
    else:
        print("⚠️  Document still exists after deletion attempt")
except Exception as e:
    print(f"❌ Failed to delete document: {e}")
    raise


🗑️  Testing: Delete the test document...
✅ Document deleted: test_setup/verification_doc
✅ Deletion verified - document no longer exists


In [ ]:
# Final verification
print("\n✅ All tests passed! Firestore is set up and working correctly.\n")

## Section 5: Summary & Next Steps

Congratulations! You've successfully set up Firestore in your GCP project.

In [ ]:
# Display summary
print("="*60)
print("🎉 FIRESTORE SETUP COMPLETE")
print("="*60)
print("\n📊 Resources Created:\n")
print(f"   ✅ Firestore Database: {DATABASE_ID}")
print(f"   ✅ Location: {REGION}")
print(f"   ✅ Type: FIRESTORE_NATIVE")
print("\n🔗 Quick Links:\n")
print(f"   📂 Firestore Console:")
print(f"      https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
print(f"\n   📊 Firestore Data:")
print(f"      https://console.cloud.google.com/firestore/data?project={PROJECT_ID}")
print(f"\n   📈 Firestore Usage:")
print(f"      https://console.cloud.google.com/firestore/usage?project={PROJECT_ID}")
print("\n💡 Next Steps:\n")
print("   1. Start adding collections and documents to your database")
print("   2. Explore Firestore features like real-time listeners")
print("   3. Set up security rules for production use")
print("   4. Consider integrating with Firebase for mobile apps")
print("\n📚 Useful Documentation:\n")
print("   - Firestore Quickstart: https://firebase.google.com/docs/firestore/quickstart")
print("   - Data Model: https://firebase.google.com/docs/firestore/data-model")
print("   - Security Rules: https://firebase.google.com/docs/firestore/security/get-started")
print("   - Python Client: https://cloud.google.com/python/docs/reference/firestore/latest")
print("\n💰 Cost Information:\n")
print("   Firestore Free Tier (per day):")
print("   - 1 GB storage")
print("   - 50,000 document reads")
print("   - 20,000 document writes")
print("   - 20,000 document deletes")
print("\n   Full pricing: https://firebase.google.com/pricing")
print("\n⚠️  Cleanup Instructions:\n")
print("   To delete the Firestore database:")
print("   1. Go to: https://console.cloud.google.com/firestore/databases")
print("   2. Select your database")
print("   3. Click 'Delete database'")
print("\n   Or use gcloud CLI:")
print(f"   gcloud firestore databases delete {DATABASE_ID} --project={PROJECT_ID}")
print("\n" + "="*60)

## Example: Basic Firestore Operations

Here are some common Firestore operations you can try:

In [ ]:
# Example: Create a collection with multiple documents
print("📝 Example: Creating sample documents...\n")

# Sample data
users = [
    {"name": "Alice", "age": 30, "city": "New York"},
    {"name": "Bob", "age": 25, "city": "San Francisco"},
    {"name": "Charlie", "age": 35, "city": "Boston"}
]

# Create documents
for user in users:
    doc_ref = db.collection("example_users").document()
    doc_ref.set(user)
    print(f"   Created user: {user['name']}")

print("\n✅ Sample documents created in 'example_users' collection")

In [ ]:
# Example: Query with filters
print("\n🔍 Example: Querying users older than 28...\n")

users_ref = db.collection("example_users")
query = users_ref.where("age", ">", 28).order_by("age")

results = query.stream()
for doc in results:
    data = doc.to_dict()
    print(f"   {data['name']}: {data['age']} years old, {data['city']}")

print("\n✅ Query completed")

In [ ]:
# Example: Update a document
print("\n✏️  Example: Updating a document...\n")

# Find Bob's document
users_ref = db.collection("example_users")
query = users_ref.where("name", "==", "Bob").limit(1)
docs = list(query.stream())

if docs:
    doc_ref = docs[0].reference
    doc_ref.update({"age": 26, "city": "Seattle"})
    print("   Updated Bob's information")
    
    # Read it back
    updated_doc = doc_ref.get()
    data = updated_doc.to_dict()
    print(f"   New data: {data['name']}, {data['age']} years old, {data['city']}")

print("\n✅ Update completed")

In [ ]:
# Example: Batch operations
print("\n🔄 Example: Batch write operations...\n")

batch = db.batch()

# Create references
ref1 = db.collection("example_batch").document("doc1")
ref2 = db.collection("example_batch").document("doc2")
ref3 = db.collection("example_batch").document("doc3")

# Add operations to batch
batch.set(ref1, {"content": "First document", "order": 1})
batch.set(ref2, {"content": "Second document", "order": 2})
batch.set(ref3, {"content": "Third document", "order": 3})

# Commit batch
batch.commit()

print("   Created 3 documents in a single batch operation")
print("\n✅ Batch operation completed")

In [ ]:
# Cleanup example data (optional)
print("\n🗑️  Cleaning up example data...\n")

# Delete example_users collection
users_ref = db.collection("example_users")
delete_count = 0
for doc in users_ref.stream():
    doc.reference.delete()
    delete_count += 1

print(f"   Deleted {delete_count} documents from example_users")

# Delete example_batch collection
batch_ref = db.collection("example_batch")
delete_count = 0
for doc in batch_ref.stream():
    doc.reference.delete()
    delete_count += 1

print(f"   Deleted {delete_count} documents from example_batch")
print("\n✅ Example data cleaned up")

## Section 6: CloudSQL Metadata Storage

This section demonstrates how to extract metadata from CloudSQL PostgreSQL and store it in Firestore using a hierarchical schema.

**What this does:**
- Connects to the CloudSQL PostgreSQL instance created in `setup_cloudsql_census.ipynb`
- Extracts table and column metadata from the `census_residence_type` table
- Stores metadata in Firestore using a hierarchical collection/subcollection pattern

**Firestore Schema:**
- Collection: `datasets` → Document: `census_data`
- Subcollection: `tables` → Document: `census_residence_type`
- Fields: Array of column definitions with metadata

In [16]:
# CloudSQL Configuration
INSTANCE_NAME = "census-demo-db"
DATABASE_NAME = "census_data"
DB_USER = "postgres"
DB_PASSWORD = "Census2021Demo!"  # Same password from setup_cloudsql_census.ipynb

# Build connection name
connection_name = f"{PROJECT_ID}:{REGION}:{INSTANCE_NAME}"

print("🔧 CloudSQL Configuration:")
print(f"  Instance: {INSTANCE_NAME}")
print(f"  Database: {DATABASE_NAME}")
print(f"  Connection Name: {connection_name}")
print(f"  Region: {REGION}")

🔧 CloudSQL Configuration:
  Instance: census-demo-db
  Database: census_data
  Connection Name: johnswain-1200-20260303091357:us-central1:census-demo-db
  Region: us-central1


In [17]:
# Install CloudSQL libraries
print("📦 Installing CloudSQL connector libraries...")
packages = [
    "cloud-sql-python-connector",
    "pg8000",
    "sqlalchemy",
    "certifi"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ CloudSQL libraries installed")

📦 Installing CloudSQL connector libraries...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ CloudSQL libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [18]:
# Import CloudSQL libraries and configure SSL
import os
import ssl
import certifi
from google.cloud.sql.connector import Connector
import sqlalchemy
from sqlalchemy import text
import aiohttp

# Configure SSL to use certifi's certificate bundle
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Create SSL context for aiohttp (used by Cloud SQL Connector)
ssl_context = ssl.create_default_context(cafile=certifi.where())

# Monkey-patch aiohttp's TCPConnector to use our SSL context
original_tcp_connector_init = aiohttp.TCPConnector.__init__

def patched_tcp_connector_init(self, *args, **kwargs):
    if 'ssl' not in kwargs or kwargs['ssl'] is None:
        kwargs['ssl'] = ssl_context
    original_tcp_connector_init(self, *args, **kwargs)

aiohttp.TCPConnector.__init__ = patched_tcp_connector_init

print("✅ CloudSQL libraries imported and SSL configured")
print(f"   SSL certificate bundle: {certifi.where()}")

✅ CloudSQL libraries imported and SSL configured
   SSL certificate bundle: /Users/johnswain/Development/gcp-demo-dataplex/venv/lib/python3.11/site-packages/certifi/cacert.pem


In [37]:
# Connect to CloudSQL PostgreSQL
print("🔌 Connecting to CloudSQL PostgreSQL...")

try:
    # Initialize connector
    connector = Connector()

    def getconn():
        conn = connector.connect(
            connection_name,
            "pg8000",
            user=DB_USER,
            password=DB_PASSWORD,
            db=DATABASE_NAME,
            enable_iam_auth=False
        )
        return conn

    # Create engine
    engine = sqlalchemy.create_engine(
        "postgresql+pg8000://",
        creator=getconn,
    )
    
    # Test connection
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), version()"))
        row = result.fetchone()
        print(f"✅ Connected to CloudSQL PostgreSQL")
        print(f"   Database: {row[0]}")
        print(f"   Version: {row[1].split(',')[0]}")
        
except Exception as e:
    print(f"❌ Failed to connect to CloudSQL: {e}")
    print("\n💡 Make sure:")
    print("   1. The CloudSQL instance is running")
    print("   2. The setup_cloudsql_census.ipynb notebook has been executed")
    print("   3. The connection details are correct")
    raise

🔌 Connecting to CloudSQL PostgreSQL...
✅ Connected to CloudSQL PostgreSQL
   Database: census_data
   Version: PostgreSQL 15.16 on x86_64-pc-linux-gnu


## Section 7: Create Table Copy and Extract Metadata

This section creates a copy of the `census_residence_type` table and then extracts comprehensive metadata from the copy.

In [38]:
# Create a copy of the original table
print("📋 Creating copy of census_residence_type table...\n")

original_table = "census_residence_type"
table_name = "census_residence_type_copy"

try:
    with engine.connect() as conn:
        # Check if copy table already exists
        check_query = text("""
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_schema = 'public'
                AND table_name = :table_name
            )
        """)
        result = conn.execute(check_query, {"table_name": table_name})
        table_exists = result.scalar()
        
        if table_exists:
            print(f"⚠️  Table '{table_name}' already exists. Dropping and recreating...")
            conn.execute(text(f"DROP TABLE {table_name}"))
            conn.commit()
        
        # Create table copy with all data
        print(f"   Creating {table_name} from {original_table}...")
        conn.execute(text(f"CREATE TABLE {table_name} AS TABLE {original_table}"))
        conn.commit()
        
        # Add primary key constraint to the copy
        print(f"   Adding primary key constraint...")
        conn.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY (geography_code)"))
        conn.commit()
        
        # Create indexes on the copy
        print(f"   Creating indexes...")
        index_queries = [
            f"CREATE INDEX idx_geography_level_copy ON {table_name}(geography_level)",
            f"CREATE INDEX idx_total_residents_copy ON {table_name}(total_residents)",
            f"CREATE INDEX idx_geography_copy ON {table_name}(geography)"
        ]
        
        for idx_query in index_queries:
            conn.execute(text(idx_query))
        conn.commit()
        
        # Get row count
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        row_count = result.scalar()
        
        print(f"\n✅ Table copy created successfully!")
        print(f"   Table: {table_name}")
        print(f"   Rows copied: {row_count:,}")
        
except Exception as e:
    print(f"❌ Error creating table copy: {e}")
    raise

📋 Creating copy of census_residence_type table...

   Creating census_residence_type_copy from census_residence_type...
   Adding primary key constraint...
   Creating indexes...

✅ Table copy created successfully!
   Table: census_residence_type_copy
   Rows copied: 232,157


In [39]:
# Extract column metadata from the copy table
print(f"\n🔍 Extracting metadata from '{table_name}'...\n")

# Query column information
column_query = text("""
    SELECT 
        column_name,
        data_type,
        character_maximum_length,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
    AND table_name = :table_name
    ORDER BY ordinal_position
""")

with engine.connect() as conn:
    result = conn.execute(column_query, {"table_name": table_name})
    columns_info = [dict(row._mapping) for row in result]

print(f"✅ Found {len(columns_info)} columns in table '{table_name}'")


🔍 Extracting metadata from 'census_residence_type_copy'...

✅ Found 8 columns in table 'census_residence_type_copy'


In [40]:
# Get primary key information from the copy table
pk_query = text("""
    SELECT kcu.column_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu 
        ON tc.constraint_name = kcu.constraint_name
        AND tc.table_schema = kcu.table_schema
    WHERE tc.constraint_type = 'PRIMARY KEY'
    AND tc.table_schema = 'public'
    AND tc.table_name = :table_name
""")

with engine.connect() as conn:
    result = conn.execute(pk_query, {"table_name": table_name})
    primary_keys = [row[0] for row in result]

print(f"✅ Primary key columns: {', '.join(primary_keys) if primary_keys else 'None'}")

✅ Primary key columns: geography_code


In [41]:
# Get index information from the copy table
index_query = text("""
    SELECT 
        indexname,
        indexdef
    FROM pg_indexes
    WHERE schemaname = 'public'
    AND tablename = :table_name
    ORDER BY indexname
""")

with engine.connect() as conn:
    result = conn.execute(index_query, {"table_name": table_name})
    indexes_info = [{"name": row[0], "definition": row[1]} for row in result]

print(f"✅ Found {len(indexes_info)} indexes")
for idx in indexes_info:
    print(f"   - {idx['name']}")

✅ Found 4 indexes
   - census_residence_type_copy_pkey
   - idx_geography_copy
   - idx_geography_level_copy
   - idx_total_residents_copy


In [42]:
# Get table statistics from the copy table
stats_query = text(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT geography_level) as num_levels
    FROM {table_name}
""")

levels_query = text(f"""
    SELECT 
        geography_level,
        COUNT(*) as count
    FROM {table_name}
    GROUP BY geography_level
    ORDER BY count DESC
""")

with engine.connect() as conn:
    # Get row counts
    result = conn.execute(stats_query)
    stats = dict(result.fetchone()._mapping)
    
    # Get geographic levels
    result = conn.execute(levels_query)
    geographic_levels = [row[0] for row in result]

print(f"✅ Table statistics:")
print(f"   Total rows: {stats['total_rows']:,}")
print(f"   Geographic levels: {', '.join(geographic_levels)}")

✅ Table statistics:
   Total rows: 232,157
   Geographic levels: OA, LSOA, MSOA, LTLA, RGN


In [43]:
# Build field metadata with descriptions
print("\n📝 Building field metadata...\n")

# Define business descriptions for each field
field_descriptions = {
    "geography_code": "Unique geographic identifier code (ONS code)",
    "date": "Census reference date (2021-03-21)",
    "geography": "Geographic area name or display label",
    "geography_level": "Level of geographic aggregation (OA, LSOA, MSOA, LTLA, UTLA, RGN, CTRY)",
    "total_residents": "Total number of usual residents in the area",
    "household_residents": "Number of residents living in households",
    "communal_residents": "Number of residents in communal establishments (care homes, student halls, etc.)",
    "created_at": "Timestamp when the record was inserted into the database"
}

# Transform column metadata into Firestore-friendly structure
fields_metadata = []

for col in columns_info:
    col_name = col['column_name']
    data_type = col['data_type'].upper()
    
    # Map PostgreSQL types to standard format
    if data_type == 'CHARACTER VARYING':
        type_str = f"VARCHAR({col['character_maximum_length']})"
    elif data_type == 'TIMESTAMP WITHOUT TIME ZONE':
        type_str = "TIMESTAMP"
    else:
        type_str = data_type
    
    # Determine mode (REQUIRED vs NULLABLE)
    mode = "REQUIRED" if col['is_nullable'] == 'NO' else "NULLABLE"
    
    # Check if primary key
    is_pk = col_name in primary_keys
    
    field_metadata = {
        "name": col_name,
        "type": type_str,
        "mode": mode,
        "description": field_descriptions.get(col_name, f"Field: {col_name}"),
        "is_primary_key": is_pk,
        "is_sensitive": False  # None of the census fields contain PII
    }
    
    fields_metadata.append(field_metadata)
    print(f"   ✓ {col_name}: {type_str} ({mode})")

print(f"\n✅ Built metadata for {len(fields_metadata)} fields")


📝 Building field metadata...

   ✓ geography_code: VARCHAR(20) (REQUIRED)
   ✓ date: DATE (NULLABLE)
   ✓ geography: VARCHAR(255) (NULLABLE)
   ✓ geography_level: VARCHAR(10) (NULLABLE)
   ✓ total_residents: INTEGER (NULLABLE)
   ✓ household_residents: INTEGER (NULLABLE)
   ✓ communal_residents: INTEGER (NULLABLE)
   ✓ created_at: TIMESTAMP (NULLABLE)

✅ Built metadata for 8 fields


## Section 8: Store Metadata in Firestore

This section writes the extracted CloudSQL metadata to Firestore using a hierarchical collection/subcollection pattern that mirrors the BigQuery hierarchy.

In [44]:
# Prepare dataset document
print("📦 Preparing dataset document...\n")

dataset_doc = {
    "database_name": DATABASE_NAME,
    "source_type": "cloudsql_postgresql",
    "instance_name": INSTANCE_NAME,
    "connection_name": connection_name,
    "region": REGION,
    "description": "UK Census 2021 demographic data - Number of usual residents in households and communal establishments",
    "labels": {
        "data_source": "uk_ons_census",
        "year": "2021",
        "survey": "TS001"
    },
    "last_sync": firestore.SERVER_TIMESTAMP
}

print("Dataset document:")
for key, value in dataset_doc.items():
    if key != "last_sync":
        print(f"   {key}: {value}")

📦 Preparing dataset document...

Dataset document:
   database_name: census_data
   source_type: cloudsql_postgresql
   instance_name: census-demo-db
   connection_name: johnswain-1200-20260303091357:us-central1:census-demo-db
   region: us-central1
   description: UK Census 2021 demographic data - Number of usual residents in households and communal establishments
   labels: {'data_source': 'uk_ons_census', 'year': '2021', 'survey': 'TS001'}


In [45]:
# Prepare table document
print("\n📊 Preparing table document...\n")

table_doc = {
    "tableName": table_name,
    "description": "Census 2021 TS001 - Copy of census_residence_type table with number of usual residents in households and communal establishments. Contains demographic data across 7 geographic levels (OA, LSOA, MSOA, LTLA, UTLA, RGN, CTRY) for England and Wales.",
    "row_count": stats['total_rows'],
    "geographic_levels": geographic_levels,
    "indexes": [
        {
            "name": idx["name"],
            "definition": idx["definition"]
        } for idx in indexes_info
    ],
    "fields": fields_metadata,
    "partitioning": None,  # PostgreSQL table is not partitioned
    "clustering_fields": None,  # PostgreSQL table is not clustered
    "last_sync": firestore.SERVER_TIMESTAMP
}

print(f"Table document:")
print(f"   tableName: {table_doc['tableName']}")
print(f"   row_count: {table_doc['row_count']:,}")
print(f"   geographic_levels: {len(table_doc['geographic_levels'])} levels")
print(f"   indexes: {len(table_doc['indexes'])} indexes")
print(f"   fields: {len(table_doc['fields'])} columns")


📊 Preparing table document...

Table document:
   tableName: census_residence_type_copy
   row_count: 232,157
   geographic_levels: 5 levels
   indexes: 4 indexes
   fields: 8 columns


In [46]:
# Write dataset document to Firestore
print("\n💾 Writing dataset document to Firestore...\n")

try:
    dataset_ref = db.collection("datasets").document(DATABASE_NAME)
    dataset_ref.set(dataset_doc, merge=True)
    
    print(f"✅ Dataset document written:")
    print(f"   Path: datasets/{DATABASE_NAME}")
    print(f"   Source: CloudSQL PostgreSQL")
    print(f"   Instance: {INSTANCE_NAME}")
    
except Exception as e:
    print(f"❌ Failed to write dataset document: {e}")
    raise


💾 Writing dataset document to Firestore...

✅ Dataset document written:
   Path: datasets/census_data
   Source: CloudSQL PostgreSQL
   Instance: census-demo-db


In [28]:
# Write table document to Firestore (in subcollection)
print("\n💾 Writing table document to Firestore...\n")

try:
    table_ref = db.collection("datasets").document(DATABASE_NAME).collection("tables").document(table_name)
    table_ref.set(table_doc, merge=True)
    
    print(f"✅ Table document written:")
    print(f"   Path: datasets/{DATABASE_NAME}/tables/{table_name}")
    print(f"   Columns: {len(fields_metadata)}")
    print(f"   Rows: {stats['total_rows']:,}")
    print(f"   Indexes: {len(indexes_info)}")
    
except Exception as e:
    print(f"❌ Failed to write table document: {e}")
    raise

print("\n" + "="*60)
print("✅ METADATA STORAGE COMPLETE")
print("="*60)
print(f"\nStored metadata in Firestore:")
print(f"   Collection: datasets")
print(f"   Dataset: {DATABASE_NAME}")
print(f"   Table: {table_name}")
print(f"   Fields: {len(fields_metadata)} columns with full metadata")
print("="*60)


💾 Writing table document to Firestore...

✅ Table document written:
   Path: datasets/census_data/tables/census_residence_type_copy
   Columns: 8
   Rows: 232,157
   Indexes: 4

✅ METADATA STORAGE COMPLETE

Stored metadata in Firestore:
   Collection: datasets
   Dataset: census_data
   Table: census_residence_type
   Fields: 8 columns with full metadata


## Section 9: Query and Validate Stored Metadata

This section demonstrates how to query the stored metadata and validates that everything was written correctly.

In [29]:
# Read back dataset document
print("🔍 Reading back dataset metadata...\n")

try:
    dataset_ref = db.collection("datasets").document(DATABASE_NAME)
    dataset_doc_read = dataset_ref.get()
    
    if dataset_doc_read.exists:
        dataset_data = dataset_doc_read.to_dict()
        print("✅ Dataset document retrieved:")
        print(f"   Database: {dataset_data.get('database_name')}")
        print(f"   Source Type: {dataset_data.get('source_type')}")
        print(f"   Instance: {dataset_data.get('instance_name')}")
        print(f"   Region: {dataset_data.get('region')}")
        print(f"   Description: {dataset_data.get('description')[:80]}...")
        print(f"   Last Sync: {dataset_data.get('last_sync')}")
    else:
        print("❌ Dataset document not found!")
        
except Exception as e:
    print(f"❌ Error reading dataset: {e}")

🔍 Reading back dataset metadata...

✅ Dataset document retrieved:
   Database: census_data
   Source Type: cloudsql_postgresql
   Instance: census-demo-db
   Region: us-central1
   Description: UK Census 2021 demographic data - Number of usual residents in households and co...
   Last Sync: 2026-03-09 19:48:21.880000+00:00


In [30]:
# Read back table document
print("\n🔍 Reading back table metadata...\n")

try:
    table_ref = db.collection("datasets").document(DATABASE_NAME).collection("tables").document(table_name)
    table_doc_read = table_ref.get()
    
    if table_doc_read.exists:
        table_data = table_doc_read.to_dict()
        print("✅ Table document retrieved:")
        print(f"   Table Name: {table_data.get('tableName')}")
        print(f"   Row Count: {table_data.get('row_count'):,}")
        print(f"   Geographic Levels: {', '.join(table_data.get('geographic_levels', []))}")
        print(f"   Number of Fields: {len(table_data.get('fields', []))}")
        print(f"   Number of Indexes: {len(table_data.get('indexes', []))}")
        print(f"   Last Sync: {table_data.get('last_sync')}")
    else:
        print("❌ Table document not found!")
        
except Exception as e:
    print(f"❌ Error reading table: {e}")


🔍 Reading back table metadata...

✅ Table document retrieved:
   Table Name: census_residence_type_copy
   Row Count: 232,157
   Geographic Levels: OA, LSOA, MSOA, LTLA, RGN
   Number of Fields: 8
   Number of Indexes: 4
   Last Sync: 2026-03-09 19:48:48.655000+00:00


In [31]:
# Display field metadata
print("\n📋 Field Metadata:\n")

if table_doc_read.exists:
    fields = table_data.get('fields', [])
    
    for field in fields:
        pk_indicator = " [PRIMARY KEY]" if field.get('is_primary_key') else ""
        print(f"   • {field['name']}: {field['type']} ({field['mode']}){pk_indicator}")
        print(f"     Description: {field['description']}")
        print()


📋 Field Metadata:

   • geography_code: VARCHAR(20) (REQUIRED) [PRIMARY KEY]
     Description: Unique geographic identifier code (ONS code)

   • date: DATE (NULLABLE)
     Description: Census reference date (2021-03-21)

   • geography: VARCHAR(255) (NULLABLE)
     Description: Geographic area name or display label

   • geography_level: VARCHAR(10) (NULLABLE)
     Description: Level of geographic aggregation (OA, LSOA, MSOA, LTLA, UTLA, RGN, CTRY)

   • total_residents: INTEGER (NULLABLE)
     Description: Total number of usual residents in the area

   • household_residents: INTEGER (NULLABLE)
     Description: Number of residents living in households

   • communal_residents: INTEGER (NULLABLE)
     Description: Number of residents in communal establishments (care homes, student halls, etc.)

   • created_at: TIMESTAMP (NULLABLE)
     Description: Timestamp when the record was inserted into the database



In [32]:
# Example Query 1: Get all tables in a dataset
print("\n" + "="*60)
print("📊 Example Query 1: Get all tables in census_data dataset")
print("="*60 + "\n")

try:
    tables_ref = db.collection("datasets").document(DATABASE_NAME).collection("tables")
    tables = tables_ref.stream()
    
    table_count = 0
    for table in tables:
        table_count += 1
        table_info = table.to_dict()
        print(f"   Table: {table.id}")
        print(f"      Rows: {table_info.get('row_count', 'N/A'):,}")
        print(f"      Fields: {len(table_info.get('fields', []))}")
        print()
    
    print(f"✅ Found {table_count} table(s) in dataset '{DATABASE_NAME}'")
    
except Exception as e:
    print(f"❌ Error querying tables: {e}")


📊 Example Query 1: Get all tables in census_data dataset

   Table: census_residence_type_copy
      Rows: 232,157
      Fields: 8

✅ Found 1 table(s) in dataset 'census_data'


In [33]:
# Example Query 2: Collection Group Query - Find all tables with a specific field
print("\n" + "="*60)
print("📊 Example Query 2: Find all tables with 'geography_code' field")
print("="*60 + "\n")

print("This demonstrates Collection Group Query capability:")
print("Searching across ALL datasets for tables containing 'geography_code'")
print()

try:
    # Collection Group Query searches across all 'tables' subcollections
    tables_ref = db.collection_group("tables")
    
    # Note: To search by field name, we'd need to structure the data differently
    # For now, we'll just show all tables and check manually
    all_tables = tables_ref.stream()
    
    matching_tables = []
    for table in all_tables:
        table_data = table.to_dict()
        fields = table_data.get('fields', [])
        
        # Check if geography_code exists in fields
        has_geography_code = any(f['name'] == 'geography_code' for f in fields)
        
        if has_geography_code:
            # Get parent dataset ID
            dataset_id = table.reference.parent.parent.id
            matching_tables.append({
                'dataset': dataset_id,
                'table': table.id,
                'row_count': table_data.get('row_count', 0)
            })
    
    print(f"✅ Found {len(matching_tables)} table(s) with 'geography_code' field:\n")
    for match in matching_tables:
        print(f"   Dataset: {match['dataset']}")
        print(f"   Table: {match['table']}")
        print(f"   Rows: {match['row_count']:,}")
        print()
    
except Exception as e:
    print(f"❌ Error in collection group query: {e}")


📊 Example Query 2: Find all tables with 'geography_code' field

This demonstrates Collection Group Query capability:
Searching across ALL datasets for tables containing 'geography_code'

✅ Found 1 table(s) with 'geography_code' field:

   Dataset: census_data
   Table: census_residence_type_copy
   Rows: 232,157



In [34]:
# Example Query 3: Find tables by source type
print("\n" + "="*60)
print("📊 Example Query 3: Find all CloudSQL PostgreSQL datasets")
print("="*60 + "\n")

try:
    datasets_ref = db.collection("datasets")
    query = datasets_ref.where("source_type", "==", "cloudsql_postgresql")
    
    results = query.stream()
    
    dataset_count = 0
    for dataset in results:
        dataset_count += 1
        dataset_info = dataset.to_dict()
        print(f"   Dataset: {dataset.id}")
        print(f"      Instance: {dataset_info.get('instance_name')}")
        print(f"      Region: {dataset_info.get('region')}")
        print(f"      Description: {dataset_info.get('description', 'N/A')[:60]}...")
        print()
    
    print(f"✅ Found {dataset_count} CloudSQL PostgreSQL dataset(s)")
    
except Exception as e:
    print(f"❌ Error querying datasets: {e}")


📊 Example Query 3: Find all CloudSQL PostgreSQL datasets

   Dataset: census_data
      Instance: census-demo-db
      Region: us-central1
      Description: UK Census 2021 demographic data - Number of usual residents ...

✅ Found 1 CloudSQL PostgreSQL dataset(s)


/Users/johnswain/Development/gcp-demo-dataplex/venv/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


In [35]:
# Close CloudSQL connections
print("\n🔌 Closing CloudSQL connections...\n")

try:
    engine.dispose()
    connector.close()
    print("✅ CloudSQL connections closed")
except Exception as e:
    print(f"⚠️  Error closing connections: {e}")

print("\n" + "="*60)
print("✅ CLOUDSQL METADATA INTEGRATION COMPLETE")
print("="*60)
print("\nYou now have CloudSQL table metadata stored in Firestore!")
print(f"\n📊 Table Created and Cataloged:")
print(f"   CloudSQL Table: census_residence_type_copy")
print(f"   Firestore Path: datasets/census_data/tables/census_residence_type_copy")
print(f"   Rows: {stats['total_rows']:,}")
print("\n💡 Next Steps:")
print("   1. Add more tables from CloudSQL by modifying the extraction queries")
print("   2. Integrate BigQuery table metadata using the same pattern")
print("   3. Build a search interface to query across all data sources")
print("   4. Add custom metadata fields (data quality, PII tags, etc.)")
print("\n📚 Firestore Console:")
print(f"   https://console.cloud.google.com/firestore/data?project={PROJECT_ID}")
print("="*60)


🔌 Closing CloudSQL connections...

✅ CloudSQL connections closed

✅ CLOUDSQL METADATA INTEGRATION COMPLETE

You now have CloudSQL table metadata stored in Firestore!

💡 Next Steps:
   1. Add more tables from CloudSQL by modifying the extraction queries
   2. Integrate BigQuery table metadata using the same pattern
   3. Build a search interface to query across all data sources
   4. Add custom metadata fields (data quality, PII tags, etc.)

📚 Firestore Console:
   https://console.cloud.google.com/firestore/data?project=johnswain-1200-20260303091357
